# Part 2 : Streaming Application (2.1.1)

In [1]:
import sys
sys.path.append("../")

In [2]:
from time import sleep
import config
import csv
from utils import publish_message, connect_kafka_producer
from datetime import datetime

## 2.1.1 Producer Implementation

Traffic camera datasets are used to simulate real-time vehicle detections. Each camera acts as an independent Kafka producer and continuously publishes batches of vehicle events into dedicated Kafka topics.

Each published batch contains:

* Camera metadata
* Producer identifier
* A collection of vehicle detection events

Each event includes:

* Vehicle plate number
* Timestamp
* Speed reading
* Event identifier

The producers publish records incrementally at configurable intervals to emulate live traffic camera feeds.

In [ ]:
if __name__ == '__main__':
    try:
        # Kafka topic for Camera C
        topic = 'camera-events-C'

        print('Publishing records...')

        # Create Kafka producer connection
        camera_producer_C = connect_kafka_producer()

        # Event fields to include in each message
        event_cols = ["event_id", "car_plate", "timestamp", "speed_reading"]

        # Read generated camera event dataset
        with open("../data/camera_event_C.csv", "r") as f:
            reader = csv.DictReader(f)

            # Initialise first batch
            current_batch_id = 1
            current_batch = {
                "batch_id": current_batch_id,
                "camera_id": 3,
                "producer_id": "C",
                "events": []
            }

            # Stream records batch by batch
            for row in reader:

                # Add event into current batch
                if int(row["batch_id"]) == current_batch_id:
                    current_batch["events"].append({
                        **{key: row[key] for key in event_cols},
                        "producer_sent_time": datetime.utcnow().isoformat()
                    })

                else:
                    # Simulate real-time publishing delay
                    sleep(config.BATCH_PUBLISH_RATE_C)

                    # Publish completed batch to Kafka
                    publish_message(
                        camera_producer_C,
                        topic,
                        current_batch
                    )

                    # Move to next batch
                    current_batch_id += 1

                    # Create new batch structure
                    current_batch = {
                        "batch_id": current_batch_id,
                        "camera_id": 3,
                        "producer_id": "C",
                        "events": []
                    }

                    # Add current row into new batch
                    current_batch["events"].append(
                        {key: row[key] for key in event_cols}
                    )

        print("Streaming completed successfully.")

    except Exception as ex:
        print('Exception while streaming producer C')
        print(str(ex))

Publishing records...
Exception in publishing message.
KafkaTimeoutError: Failed to update metadata after 60.0 secs.
Exception in publishing message.
KafkaTimeoutError: Failed to update metadata after 60.0 secs.
